# 02 - Data preparation and series selection

This notebook continues the verified raw-data audit in `01_data_audit_and_eda.ipynb`. It prepares a small, documented group of product-store series for time-series analysis and forecasting.

The work has four goals:

1. keep the final 28 days out of selection and development data;
2. create chronological training, validation, and test periods;
3. select series that represent different demand patterns, stores, states, and categories; and
4. reshape and join the selected sales without changing the number of observations.

The M5 values are recorded sales. A zero means that no sale was recorded on that day. It does not tell us whether customer demand was zero or the item was unavailable.

## 1. Imports and project paths

The notebook uses project-relative paths, so it can run from the repository root or from the `notebooks` folder. Figures used in the report are saved under `reports/figures/series_eda`. Generated analysis data are saved under `data/processed` and remain excluded from Git.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "series_eda"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed-data directory: {PROCESSED_DATA_DIR}")
print(f"Report-figure directory: {FIGURE_DIR}")

Project root: /Users/sumitchaurasia/Desktop/USD/FinalProj/module_one_project
Processed-data directory: /Users/sumitchaurasia/Desktop/USD/FinalProj/module_one_project/data/processed
Report-figure directory: /Users/sumitchaurasia/Desktop/USD/FinalProj/module_one_project/reports/figures/series_eda


## 2. Verify the source files

The download helper checks each required file against its expected checksum. This prevents the analysis from continuing with an incomplete or changed copy of the M5 data.

In [2]:
from scripts.download_data import REQUIRED_FILE_HASHES, invalid_files

source_file_check = pd.DataFrame(
    {
        "file": list(REQUIRED_FILE_HASHES),
        "exists": [(RAW_DATA_DIR / name).is_file() for name in REQUIRED_FILE_HASHES],
        "size_mb": [
            round((RAW_DATA_DIR / name).stat().st_size / (1024**2), 2)
            if (RAW_DATA_DIR / name).is_file()
            else 0
            for name in REQUIRED_FILE_HASHES
        ],
    }
)

display(source_file_check)
assert not invalid_files(RAW_DATA_DIR), "A required M5 file is missing or failed checksum verification."
print("All five source files passed checksum verification.")

,file,exists,size_mb
0,calendar.csv,True,0.10
1,sales_train_evaluation.csv,True,116.10
2,sales_train_validation.csv,True,114.45
3,sample_submission.csv,True,4.99
4,sell_prices.csv,True,193.97


All five source files passed checksum verification.


## 3. Define the chronological split

The M5 evaluation history contains 1,941 days. We use two 28-day periods at the end:

- training: `d_1` to `d_1885`;
- validation: `d_1886` to `d_1913`; and
- final test: `d_1914` to `d_1941`.

The validation period will support model choice and tuning. The final test period stays untouched until the project reaches final model and inventory-policy evaluation. This notebook reads sales only through `d_1913`.

In [3]:
TRAIN_END = 1885
VALIDATION_START = 1886
VALIDATION_END = 1913
TEST_START = 1914
TEST_END = 1941

calendar = pd.read_csv(RAW_DATA_DIR / "calendar.csv", parse_dates=["date"])

split_summary = pd.DataFrame(
    {
        "split": ["Training", "Validation", "Final test"],
        "first_day": [f"d_1", f"d_{VALIDATION_START}", f"d_{TEST_START}"],
        "last_day": [f"d_{TRAIN_END}", f"d_{VALIDATION_END}", f"d_{TEST_END}"],
        "days": [TRAIN_END, VALIDATION_END - VALIDATION_START + 1, TEST_END - TEST_START + 1],
    }
)

date_lookup = calendar.set_index("d")["date"]
split_summary["first_date"] = split_summary["first_day"].map(date_lookup).dt.date
split_summary["last_date"] = split_summary["last_day"].map(date_lookup).dt.date

display(split_summary)
assert split_summary["days"].sum() == TEST_END
assert VALIDATION_END + 1 == TEST_START
print("The periods are chronological and do not overlap.")

,split,first_day,last_day,days,first_date,last_date
0,Training,d_1,d_1885,1885,2011-01-29,2016-03-27
1,Validation,d_1886,d_1913,28,2016-03-28,2016-04-24
2,Final test,d_1914,d_1941,28,2016-04-25,2016-05-22


The periods are chronological and do not overlap.


## 4. Load development-period sales and prices

Only identifiers and `d_1` through `d_1913` are loaded from the evaluation sales file. The final test values are therefore unavailable to the selection and EDA code below.

Daily sales use a smaller integer type to reduce memory use. The price table is loaded once because it will be filtered to the selected product-store combinations before the join.

In [4]:
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
train_day_columns = [f"d_{day}" for day in range(1, TRAIN_END + 1)]
validation_day_columns = [f"d_{day}" for day in range(VALIDATION_START, VALIDATION_END + 1)]
development_day_columns = train_day_columns + validation_day_columns

evaluation_header = pd.read_csv(RAW_DATA_DIR / "sales_train_evaluation.csv", nrows=0)
available_day_columns = [name for name in evaluation_header.columns if name.startswith("d_")]

assert available_day_columns[-1] == f"d_{TEST_END}"
assert set(development_day_columns).issubset(available_day_columns)

sales_dtypes = {name: "int16" for name in development_day_columns}
sales_dtypes.update({name: "category" for name in identifier_columns})

sales_development = pd.read_csv(
    RAW_DATA_DIR / "sales_train_evaluation.csv",
    usecols=identifier_columns + development_day_columns,
    dtype=sales_dtypes,
)

price_dtypes = {
    "store_id": "category",
    "item_id": "category",
    "wm_yr_wk": "int32",
    "sell_price": "float32",
}
sell_prices = pd.read_csv(RAW_DATA_DIR / "sell_prices.csv", dtype=price_dtypes)

print(f"Development sales shape: {sales_development.shape}")
print(f"Loaded sales days: {development_day_columns[0]} to {development_day_columns[-1]}")
print(f"Selling-price shape: {sell_prices.shape}")
assert f"d_{TEST_START}" not in sales_development.columns
print("Final test sales were not loaded.")

Development sales shape: (30490, 1919)
Loaded sales days: d_1 to d_1913
Selling-price shape: (6841121, 4)
Final test sales were not loaded.


## 5. Calculate training-only series measures

Series selection uses the training period only. Five measures describe each product-store series:

- mean daily sales gives its average recorded volume;
- zero-sales share measures intermittency;
- coefficient of variation compares variability with the series mean;
- nonzero days in the most recent training year remove inactive items from some profiles; and
- lag-7 autocorrelation measures whether sales tend to repeat after one week.

Lag-7 autocorrelation is calculated in small row batches to control memory use. A high value supports a repeated weekly pattern. It does not prove that weekday effects are the only source of seasonality.

In [5]:
training_values = sales_development[train_day_columns].to_numpy(copy=False)

mean_daily_sales = training_values.mean(axis=1)
std_daily_sales = training_values.std(axis=1, ddof=1)
zero_sales_share = (training_values == 0).mean(axis=1)
coefficient_of_variation = np.divide(
    std_daily_sales,
    mean_daily_sales,
    out=np.full(len(sales_development), np.nan),
    where=mean_daily_sales > 0,
)
recent_nonzero_days = (training_values[:, -365:] > 0).sum(axis=1)

# Compute row-wise lag-7 correlations without creating several full-size copies.
lag7_autocorrelation = np.full(len(sales_development), np.nan, dtype="float32")
batch_size = 1000

for start in range(0, len(sales_development), batch_size):
    stop = min(start + batch_size, len(sales_development))
    block = training_values[start:stop].astype("float32")
    earlier = block[:, :-7]
    later = block[:, 7:]

    earlier_mean = earlier.mean(axis=1)
    later_mean = later.mean(axis=1)
    covariance = (earlier * later).mean(axis=1) - earlier_mean * later_mean
    earlier_std = np.sqrt(np.maximum((earlier**2).mean(axis=1) - earlier_mean**2, 0))
    later_std = np.sqrt(np.maximum((later**2).mean(axis=1) - later_mean**2, 0))
    denominator = earlier_std * later_std

    lag7_autocorrelation[start:stop] = np.divide(
        covariance,
        denominator,
        out=np.full(stop - start, np.nan, dtype="float32"),
        where=denominator > 0,
    )

series_metrics = sales_development[identifier_columns].copy()
series_metrics["mean_daily_sales"] = mean_daily_sales
series_metrics["zero_sales_share"] = zero_sales_share
series_metrics["coefficient_of_variation"] = coefficient_of_variation
series_metrics["recent_nonzero_days"] = recent_nonzero_days
series_metrics["lag7_autocorrelation"] = lag7_autocorrelation

metric_summary = series_metrics[
    [
        "mean_daily_sales",
        "zero_sales_share",
        "coefficient_of_variation",
        "recent_nonzero_days",
        "lag7_autocorrelation",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9])

display(metric_summary.round(3))
assert series_metrics["id"].duplicated().sum() == 0
assert series_metrics["mean_daily_sales"].notna().all()

,mean_daily_sales,zero_sales_share,coefficient_of_variation,recent_nonzero_days,lag7_autocorrelation
count,30490.000,30490.000,30490.000,30490.000,30490.000
mean,1.122,0.684,2.414,145.607,0.210
std,2.730,0.224,1.395,92.136,0.162
min,0.001,0.002,0.254,0.000,-0.058
25%,0.183,0.539,1.463,69.000,0.080
50%,0.439,0.738,2.056,129.000,0.174
75%,1.056,0.869,2.994,212.000,0.308
90%,2.437,0.932,4.153,285.000,0.445
max,131.253,0.999,43.417,365.000,0.868


## 6. Select a reproducible group of series

The project uses 12 series. The selection plan covers all 10 stores, all three states, and all three product categories. Each profile has three series.

The rules are applied within a fixed store-category slot:

- stable: at least one mean unit per day, no more than 25% zero days, at least 180 nonzero days in the recent year, then the lowest coefficient of variation;
- weekly pattern: at least 0.5 mean units per day, no more than 40% zero days, at least 100 recent nonzero days, then the highest lag-7 autocorrelation;
- intermittent: 65% to 90% zero days and 20 to 150 recent nonzero days, then the highest zero-sales share; and
- variable: at least 0.5 mean units per day, no more than 60% zero days, at least 60 recent nonzero days, then the highest coefficient of variation.

`item_id` is the final sort field in every rule. It resolves tied measures and makes the selected list repeatable across pandas versions. The profile names describe why each series was selected. They are working labels for this project, not permanent product classifications.

In [6]:
selection_plan = pd.DataFrame(
    [
        ("stable", "CA_1", "FOODS"),
        ("stable", "TX_1", "HOUSEHOLD"),
        ("stable", "WI_1", "HOBBIES"),
        ("weekly_pattern", "CA_2", "HOUSEHOLD"),
        ("weekly_pattern", "TX_2", "HOBBIES"),
        ("weekly_pattern", "WI_2", "FOODS"),
        ("intermittent", "CA_3", "HOBBIES"),
        ("intermittent", "TX_3", "FOODS"),
        ("intermittent", "WI_3", "HOUSEHOLD"),
        ("variable", "CA_4", "FOODS"),
        ("variable", "TX_1", "HOBBIES"),
        ("variable", "WI_1", "HOUSEHOLD"),
    ],
    columns=["profile", "store_id", "cat_id"],
)

display(selection_plan)

,profile,store_id,cat_id
0,stable,CA_1,FOODS
1,stable,TX_1,HOUSEHOLD
2,stable,WI_1,HOBBIES
3,weekly_pattern,CA_2,HOUSEHOLD
4,weekly_pattern,TX_2,HOBBIES
5,weekly_pattern,WI_2,FOODS
6,intermittent,CA_3,HOBBIES
7,intermittent,TX_3,FOODS
8,intermittent,WI_3,HOUSEHOLD
9,variable,CA_4,FOODS


In [7]:
selected_rows = []

for plan_row in selection_plan.itertuples(index=False):
    candidates = series_metrics[
        (series_metrics["store_id"].astype(str) == plan_row.store_id)
        & (series_metrics["cat_id"].astype(str) == plan_row.cat_id)
    ].copy()

    if plan_row.profile == "stable":
        candidates = candidates[
            (candidates["mean_daily_sales"] >= 1.0)
            & (candidates["zero_sales_share"] <= 0.25)
            & (candidates["recent_nonzero_days"] >= 180)
        ].sort_values(
            ["coefficient_of_variation", "zero_sales_share", "item_id"],
            ascending=[True, True, True],
        )
    elif plan_row.profile == "weekly_pattern":
        candidates = candidates[
            (candidates["mean_daily_sales"] >= 0.5)
            & (candidates["zero_sales_share"] <= 0.40)
            & (candidates["recent_nonzero_days"] >= 100)
        ].sort_values(
            ["lag7_autocorrelation", "zero_sales_share", "item_id"],
            ascending=[False, True, True],
        )
    elif plan_row.profile == "intermittent":
        candidates = candidates[
            candidates["zero_sales_share"].between(0.65, 0.90)
            & candidates["recent_nonzero_days"].between(20, 150)
        ].sort_values(
            ["zero_sales_share", "coefficient_of_variation", "item_id"],
            ascending=[False, False, True],
        )
    else:
        candidates = candidates[
            (candidates["mean_daily_sales"] >= 0.5)
            & (candidates["zero_sales_share"] <= 0.60)
            & (candidates["recent_nonzero_days"] >= 60)
        ].sort_values(
            ["coefficient_of_variation", "zero_sales_share", "item_id"],
            ascending=[False, False, True],
        )

    assert not candidates.empty, f"No eligible series for {plan_row}"
    selected_row = candidates.iloc[0].copy()
    selected_row["profile"] = plan_row.profile
    selected_rows.append(selected_row)

selected_metrics = pd.DataFrame(selected_rows).reset_index(drop=True)
selected_metrics["series_key"] = (
    selected_metrics["item_id"].astype(str) + "_" + selected_metrics["store_id"].astype(str)
)

selected_display_columns = [
    "profile",
    "store_id",
    "state_id",
    "cat_id",
    "item_id",
    "mean_daily_sales",
    "zero_sales_share",
    "coefficient_of_variation",
    "recent_nonzero_days",
    "lag7_autocorrelation",
]

display(selected_metrics[selected_display_columns].round(3))

assert len(selected_metrics) == 12
assert selected_metrics[["item_id", "store_id"]].duplicated().sum() == 0
assert selected_metrics["store_id"].nunique() == 10
assert selected_metrics["state_id"].nunique() == 3
assert selected_metrics["cat_id"].nunique() == 3
assert selected_metrics.groupby("profile").size().eq(3).all()

,profile,store_id,state_id,cat_id,item_id,mean_daily_sales,zero_sales_share,coefficient_of_variation,recent_nonzero_days,lag7_autocorrelation
0,stable,CA_1,CA,FOODS,FOODS_3_586,45.950,0.004,0.358,364,0.591
1,stable,TX_1,TX,HOUSEHOLD,HOUSEHOLD_1_373,6.858,0.035,0.578,352,0.395
2,stable,WI_1,WI,HOBBIES,HOBBIES_1_345,3.608,0.136,0.739,335,0.246
3,weekly_pattern,CA_2,CA,HOUSEHOLD,HOUSEHOLD_1_191,7.016,0.299,1.046,283,0.663
4,weekly_pattern,TX_2,TX,HOBBIES,HOBBIES_1_323,4.334,0.077,0.665,314,0.438
5,weekly_pattern,WI_2,WI,FOODS,FOODS_1_096,14.941,0.191,1.129,352,0.784
6,intermittent,CA_3,CA,HOBBIES,HOBBIES_2_110,0.150,0.900,3.474,55,0.028
7,intermittent,TX_3,TX,FOODS,FOODS_2_254,0.134,0.900,3.343,66,0.149
8,intermittent,WI_3,WI,HOUSEHOLD,HOUSEHOLD_1_161,0.118,0.900,3.214,103,0.155
9,variable,CA_4,CA,FOODS,FOODS_2_255,0.919,0.592,2.181,161,0.179


### Selection inference

The assertions confirm that the sample has 12 distinct product-store series, three series per profile, all three states, all three categories, and all 10 stores. The selected metrics should remain separated by profile: stable series should have lower variation and fewer zero days, intermittent series should have many zero days, weekly-pattern series should have stronger lag-7 correlation, and variable series should have higher coefficients of variation.

This is a purposive sample. It supports comparisons across several demand behaviors, but its results will not estimate performance across all 30,490 M5 series.

## 7. Reshape and join calendar and price data

The selected sales are reshaped from one row per series to one row per series-day. Calendar fields are joined by `d`. Weekly prices are joined by store, item, and Walmart year-week.

Both joins use `validate="many_to_one"`. The notebook also checks row counts and unique series-date keys. These checks would stop execution if a duplicate lookup key multiplied the sales rows.

In [8]:
selected_keys = selected_metrics[["item_id", "store_id"]].astype(str)
sales_key = sales_development["item_id"].astype(str) + "|" + sales_development["store_id"].astype(str)
selected_key_values = set(selected_keys["item_id"] + "|" + selected_keys["store_id"])

selected_sales = sales_development[sales_key.isin(selected_key_values)].copy()
selected_sales["item_id"] = selected_sales["item_id"].astype(str)
selected_sales["store_id"] = selected_sales["store_id"].astype(str)

development_long = selected_sales.melt(
    id_vars=identifier_columns,
    value_vars=development_day_columns,
    var_name="d",
    value_name="sales",
)
development_long["day_number"] = development_long["d"].str.removeprefix("d_").astype("int16")
development_long["split"] = np.where(
    development_long["day_number"] <= TRAIN_END,
    "train",
    "validation",
)

rows_before_joins = len(development_long)

calendar_columns = [
    "d",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]

development_long = development_long.merge(
    calendar[calendar_columns],
    on="d",
    how="left",
    validate="many_to_one",
)

# Keep only price rows that can match one of the selected item-store pairs.
price_keys = selected_metrics[["item_id", "store_id"]].astype(str).drop_duplicates()
price_for_selected = sell_prices.copy()
price_for_selected["item_id"] = price_for_selected["item_id"].astype(str)
price_for_selected["store_id"] = price_for_selected["store_id"].astype(str)
price_for_selected = price_for_selected.merge(
    price_keys,
    on=["item_id", "store_id"],
    how="inner",
    validate="many_to_one",
)

development_long = development_long.merge(
    price_for_selected[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one",
)

development_long = development_long.merge(
    selected_metrics[["item_id", "store_id", "profile", "series_key"]].astype(
        {"item_id": str, "store_id": str}
    ),
    on=["item_id", "store_id"],
    how="left",
    validate="many_to_one",
)

join_audit = pd.DataFrame(
    {
        "check": [
            "Expected development rows",
            "Rows before joins",
            "Rows after joins",
            "Duplicate series-date rows",
            "Missing dates",
            "Missing profile labels",
            "Missing prices",
            "Missing prices with positive sales",
            "Final-test rows present",
        ],
        "result": [
            len(selected_metrics) * len(development_day_columns),
            rows_before_joins,
            len(development_long),
            development_long.duplicated(["item_id", "store_id", "date"]).sum(),
            development_long["date"].isna().sum(),
            development_long["profile"].isna().sum(),
            development_long["sell_price"].isna().sum(),
            (
                development_long["sell_price"].isna()
                & development_long["sales"].gt(0)
            ).sum(),
            development_long["day_number"].ge(TEST_START).sum(),
        ],
    }
)

display(join_audit)

assert len(development_long) == rows_before_joins
assert development_long.duplicated(["item_id", "store_id", "date"]).sum() == 0
assert development_long["date"].notna().all()
assert development_long["profile"].notna().all()
assert development_long["day_number"].max() == VALIDATION_END

,check,result
0,Expected development rows,22956
1,Rows before joins,22956
2,Rows after joins,22956
3,Duplicate series-date rows,0
4,Missing dates,0
5,Missing profile labels,0
6,Missing prices,2009
7,Missing prices with positive sales,0
8,Final-test rows present,0


### Join inference

Equal row counts before and after the joins show that calendar and price data did not duplicate or remove sales observations. Missing prices are kept as missing rather than filled with a made-up value. The separate count of missing prices with positive sales helps distinguish pre-release periods from gaps that could affect later price-based models.

## 8. Confirm the saved split

The processed dataset contains training and validation rows only. Final test dates are recorded in the split specification, but their sales values are not saved here.

In [9]:
split_check = (
    development_long.groupby("split", observed=True)
    .agg(
        rows=("sales", "size"),
        series=("series_key", "nunique"),
        first_day=("day_number", "min"),
        last_day=("day_number", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

display(split_check)
assert split_check.loc[split_check["split"] == "train", "rows"].iat[0] == 12 * TRAIN_END
assert split_check.loc[split_check["split"] == "validation", "rows"].iat[0] == 12 * 28
assert set(development_long["split"]) == {"train", "validation"}

,split,rows,series,first_day,last_day,first_date,last_date
0,train,22620,12,1,1885,2011-01-29,2016-03-27
1,validation,336,12,1886,1913,2016-03-28,2016-04-24


## 10. Save reproducible outputs

The metadata file records the exact selected series and the training-only measures used to choose them. The development file contains joined training and validation rows. It does not contain final test sales.

In [10]:
metadata_output = PROCESSED_DATA_DIR / "selected_series_metadata.csv"
development_output = PROCESSED_DATA_DIR / "selected_series_development.csv"

metadata_columns = [
    "series_key",
    "profile",
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "mean_daily_sales",
    "zero_sales_share",
    "coefficient_of_variation",
    "recent_nonzero_days",
    "lag7_autocorrelation",
]
selected_metrics[metadata_columns].to_csv(metadata_output, index=False)

development_output_columns = [
    "series_key",
    "profile",
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "d",
    "day_number",
    "date",
    "split",
    "sales",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "sell_price",
]
development_long[development_output_columns].sort_values(
    ["series_key", "day_number"]
).to_csv(development_output, index=False)

saved_outputs = pd.DataFrame(
    {
        "file": [metadata_output.name, development_output.name],
        "rows": [len(selected_metrics), len(development_long)],
        "size_mb": [
            round(metadata_output.stat().st_size / (1024**2), 3),
            round(development_output.stat().st_size / (1024**2), 3),
        ],
    }
)
display(saved_outputs)

assert selected_metrics[metadata_columns].shape[0] == 12
assert development_long["day_number"].max() == VALIDATION_END
print("Processed outputs saved without final-test sales.")

,file,rows,size_mb
0,selected_series_metadata.csv,12,0.002
1,selected_series_development.csv,22956,3.672


Processed outputs saved without final-test sales.


## 10. Conclusions and next step

This notebook reserves the final 28 days, defines the training and validation periods, selects 12 series with deterministic rules, validates the calendar and price joins, and saves development data through `d_1913`.

Notebook 03 reads the processed files and studies the selected time series. It does not repeat the full raw-data preparation.